In [ ]:
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Qwen3VLForConditionalGeneration.from_pretrained(
    'Qwen/Qwen3-VL-2B-Instruct',
    dtype=torch.float16,
    device_map=device
)
print("config.dtype:", model.config.dtype)
processor = AutoProcessor.from_pretrained(
    'Qwen/Qwen3-VL-2B-Instruct',
    max_pixels=1024 * 1024)

#### Info

In [ ]:
def debug_find_layers(model):
    candidates = [
        ("language_model.model.layers", lambda m: m.language_model.model.layers),
        ("language_model.layers", lambda m: m.language_model.layers),
        ("model.language_model.model.layers", lambda m: m.model.language_model.model.layers),
        ("model.language_model.layers", lambda m: m.model.language_model.layers),
        ("model.layers", lambda m: m.model.layers),
        ("layers", lambda m: m.layers),
    ]
    for name, fn in candidates:
        try:
            r = fn(model)
            if r is not None:
                print("Matched:", name, "len:", len(r), "type:", type(r))
                return r
        except AttributeError:
            continue
    raise AttributeError(f"Cannot locate transformer layers in {type(model).__name__}.")

def debug_find_visual(model):
    candidates = [
        ("model.visual", lambda m: m.model.visual),
        ("visual", lambda m: m.visual),
        ("model.vision_model", lambda m: m.model.vision_model),
        ("vision_model", lambda m: m.vision_model),
    ]
    for name, fn in candidates:
        try:
            r = fn(model)
            if r is not None:
                print("Matched:", name, "type:", type(r))
                return r
        except AttributeError:
            continue
    raise AttributeError(f"Cannot locate visual module in {type(model).__name__}.")

In [ ]:
debug_find_layers(model)

In [ ]:
debug_find_visual(model)